# 0. Overview

This notebook analyzes whether the ENSO-linked vertical circulation over the Maritime Continent changed between **Full (1981-2025)**, **Past (1981-2006)**, and **Recent (2007-2025)** periods.

Scientific objective:
- Separate the DJF seasonal mean state from the ENSO composite response in the vertical circulation field.
- Use omega (`w`) anomalies as shaded fields and overlay scaled wind vectors for the cross-section view.
- Produce both longitude-pressure and latitude-pressure composites for El Nino and La Nina.


# 1. Import Library


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['axes.titleweight'] = 'regular'
plt.rcParams['figure.dpi'] = 110

np.set_printoptions(suppress=True, linewidth=120)


# 2. Open Data & Pre-Process


#### Open ERA5 vertical wind GRIB


In [ ]:
uvw_path = '../../external/ClimateData/era5-monthly/uvw_1980-2025_1000hpa-100hpa.grib'
optional_grib_path = '../../external/ClimateData/era5-monthly/geopot_sh_temp_vort_1980-2025_1000hpa-100hpa.grib'

def open_grib_variable(path: str | Path, short_name: str) -> xr.Dataset:
    return xr.open_dataset(
        path,
        engine='cfgrib',
        backend_kwargs={
            'indexpath': '',
            'filter_by_keys': {
                'shortName': short_name,
                'typeOfLevel': 'isobaricInhPa',
            },
        },
        chunks={'time': 12},
    )


u_ds = open_grib_variable(uvw_path, 'u').rename({'latitude': 'lat', 'longitude': 'lon', 'isobaricInhPa': 'level'})
v_ds = open_grib_variable(uvw_path, 'v').rename({'latitude': 'lat', 'longitude': 'lon', 'isobaricInhPa': 'level'})
w_ds = open_grib_variable(uvw_path, 'w').rename({'latitude': 'lat', 'longitude': 'lon', 'isobaricInhPa': 'level'})

ds_uvw = xr.merge([u_ds[['u']], v_ds[['v']], w_ds[['w']]], compat='override')

if 'lon' in ds_uvw.coords:
    ds_uvw = ds_uvw.assign_coords(lon=(ds_uvw.lon % 360)).sortby('lon')
if 'lat' in ds_uvw.coords:
    ds_uvw = ds_uvw.sortby('lat')
if 'level' in ds_uvw.coords:
    ds_uvw = ds_uvw.sortby('level')

rename_map = {}
for old_name, new_name in [('latitude', 'lat'), ('longitude', 'lon'), ('isobaricInhPa', 'level')]:
    if old_name in ds_uvw.dims or old_name in ds_uvw.coords:
        rename_map[old_name] = new_name
ds_uvw = ds_uvw.rename(rename_map)

if 'lon' in ds_uvw.coords:
    ds_uvw = ds_uvw.assign_coords(lon=(ds_uvw.lon % 360)).sortby('lon')
if 'lat' in ds_uvw.coords:
    ds_uvw = ds_uvw.sortby('lat')
if 'level' in ds_uvw.coords:
    ds_uvw = ds_uvw.sortby('level')

ds_uvw = ds_uvw.sel(time=slice('1980-12-01', '2025-02-01'))
ds_uvw = ds_uvw.sel(lon=slice(60, 270))
ds_uvw = ds_uvw.sel(lat=slice(-10, 10))

ds_uvw


#### Open NINO34 index


In [ ]:
nino34_path = '../../external/ClimateData/index-monthly/nino34.anom.csv'

df_nino34 = pd.read_csv(nino34_path, parse_dates=['Date'])
nino34_column = next(col for col in df_nino34.columns if col != 'Date')
df_nino34[nino34_column] = pd.to_numeric(df_nino34[nino34_column], errors='coerce').replace(-9999.0, np.nan)
df_nino34 = df_nino34.set_index('Date').loc['1980-12-01':'2025-02-01', [nino34_column]].copy()

full_years = np.arange(1981, 2026)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2026)

df_nino34.head()


# 3. Helper Functions / settings


In [ ]:
def assign_djf_year(time_coord: xr.DataArray) -> xr.DataArray:
    return xr.where(time_coord.dt.month == 12, time_coord.dt.year + 1, time_coord.dt.year)


def compute_djf_means(obj: xr.Dataset | xr.DataArray) -> xr.Dataset | xr.DataArray:
    monthly = obj.where(obj.time.dt.month.isin([12, 1, 2]), drop=True)
    djf_year = assign_djf_year(monthly.time)
    monthly = monthly.assign_coords(djf_year=('time', djf_year.values))
    return monthly.groupby('djf_year').mean('time')


def compute_djf_index(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    seasonal = df.loc[df.index.month.isin([12, 1, 2]), [value_col]].copy()
    seasonal['djf_year'] = seasonal.index.year + (seasonal.index.month == 12).astype(int)
    out = seasonal.groupby('djf_year')[value_col].mean().to_frame(name='nino34')
    out.index.name = 'djf_year'
    return out


def select_enso_years(index_df: pd.DataFrame, threshold: float = 0.5, mode: str = 'elnino') -> list[int]:
    series = index_df['nino34']
    if mode == 'elnino':
        mask = series >= threshold
    elif mode == 'lanina':
        mask = series <= -threshold
    else:
        raise ValueError("mode must be 'elnino' or 'lanina'")
    return series.index[mask].astype(int).tolist()


def symmetric_levels(*arrays, n: int = 21, percentile: float = 98.0) -> np.ndarray:
    values = []
    for arr in arrays:
        arr_values = np.asarray(arr).ravel()
        if arr_values.size:
            values.append(arr_values)
    if not values:
        return np.linspace(-1.0, 1.0, n)
    values = np.concatenate(values)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.linspace(-1.0, 1.0, n)
    vmax = np.nanpercentile(np.abs(values), percentile)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = np.nanmax(np.abs(values))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    return np.linspace(-vmax, vmax, n)


def quiver_step(size: int, target: int) -> int:
    return max(1, int(np.ceil(size / target)))


def period_anomaly(djf_ds: xr.Dataset, years: np.ndarray) -> tuple[xr.Dataset, xr.Dataset, xr.Dataset]:
    period_ds = djf_ds.sel(djf_year=years)
    climatology = period_ds.mean('djf_year')
    anomaly = period_ds - climatology
    return period_ds, climatology, anomaly


def select_lon_pressure_section(ds: xr.Dataset) -> xr.Dataset:
    return ds.sel(lon=slice(60, 270)).sel(lat=slice(-10, 10)).mean('lat')


def select_lat_pressure_section(ds: xr.Dataset) -> xr.Dataset:
    return ds.sel(lon=slice(90, 160)).sel(lat=slice(-10, 10)).mean('lon')


def omega_scale_for_bundle(bundle: dict[str, xr.Dataset], horiz_var: str) -> float:
    horiz_values = []
    omega_values = []
    for ds in bundle.values():
        horiz_values.append(np.asarray(ds[horiz_var]).ravel())
        omega_values.append(np.asarray(ds['w']).ravel())
    horiz_values = np.concatenate(horiz_values) if horiz_values else np.array([1.0])
    omega_values = np.concatenate(omega_values) if omega_values else np.array([1.0])
    horiz_values = horiz_values[np.isfinite(horiz_values)]
    omega_values = omega_values[np.isfinite(omega_values)]
    horiz_ref = np.nanpercentile(np.abs(horiz_values), 95) if horiz_values.size else 1.0
    omega_ref = np.nanpercentile(np.abs(omega_values), 95) if omega_values.size else 1.0
    if not np.isfinite(horiz_ref) or horiz_ref == 0:
        horiz_ref = 1.0
    if not np.isfinite(omega_ref) or omega_ref == 0:
        omega_ref = 1.0
    return float(horiz_ref / omega_ref)


def plot_vertical_section(
    ax: plt.Axes,
    section_ds: xr.Dataset,
    horiz_var: str,
    title: str,
    x_dim: str,
    levels: np.ndarray,
    omega_scale: float,
    x_ticks: np.ndarray | None = None,
    x_label: str | None = None,
    quiver_scale: float = 55,
):
    x = section_ds[x_dim].values
    p = section_ds['level'].values
    w = section_ds['w'].values

    cf = ax.contourf(x, p, w, levels=levels, cmap='RdBu_r', extend='both')
    ax.contour(x, p, w, levels=[0], colors='k', linewidths=0.6, alpha=0.55)

    x_step = quiver_step(section_ds[x_dim].size, target=28)
    p_step = quiver_step(section_ds['level'].size, target=10)
    qsrc = section_ds.isel({x_dim: slice(None, None, x_step), 'level': slice(None, None, p_step)})
    ax.quiver(
        qsrc[x_dim].values,
        qsrc['level'].values,
        qsrc[horiz_var].values,
        (-qsrc['w'].values * omega_scale),
        color='k',
        scale=quiver_scale,
        width=0.0022,
        headwidth=3.5,
        headlength=4.5,
        pivot='mid',
        zorder=3,
    )

    ax.set_xlim(float(np.nanmin(x)), float(np.nanmax(x)))
    ax.set_ylim(1000, 100)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(x_label if x_label is not None else x_dim)
    ax.set_ylabel('Pressure (hPa)')
    if x_ticks is not None:
        ax.set_xticks(x_ticks)
    ax.set_yticks(np.arange(1000, 99, -100))
    ax.grid(True, linestyle='--', alpha=0.25, linewidth=0.5)
    ax.text(
        0.01,
        0.03,
        f'Quiver: {horiz_var} + (-w x {omega_scale:.0f})',
        transform=ax.transAxes,
        fontsize=9,
        ha='left',
        va='bottom',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'),
    )
    return cf


period_years = {
    'full': full_years,
    'past': past_years,
    'recent': recent_years,
}

djf_ds = compute_djf_means(ds_uvw).sortby('level')
djf_nino34 = compute_djf_index(df_nino34, nino34_column)

event_years = {'elnino': {}, 'lanina': {}}
for period_name, years in period_years.items():
    period_index = djf_nino34.loc[djf_nino34.index.isin(years)]
    event_years['elnino'][period_name] = select_enso_years(period_index, threshold=0.5, mode='elnino')
    event_years['lanina'][period_name] = select_enso_years(period_index, threshold=0.5, mode='lanina')

period_anoms = {}
period_clims = {}
for period_name, years in period_years.items():
    _, period_clim, period_anom = period_anomaly(djf_ds, years)
    period_clims[period_name] = period_clim.load()
    period_anoms[period_name] = period_anom.load()

composites = {'elnino': {}, 'lanina': {}}
for event_name in ['elnino', 'lanina']:
    for period_name in ['full', 'past', 'recent']:
        years = event_years[event_name][period_name]
        composites[event_name][period_name] = period_anoms[period_name].sel(djf_year=years).mean('djf_year').load()
    composites[event_name]['diff'] = (composites[event_name]['recent'] - composites[event_name]['past']).load()

def build_section_bundle(composite_dict: dict[str, xr.Dataset], horiz_var: str):
    lon_bundle = {period: select_lon_pressure_section(ds) for period, ds in composite_dict.items()}
    lat_bundle = {period: select_lat_pressure_section(ds) for period, ds in composite_dict.items()}
    lon_scale = omega_scale_for_bundle(lon_bundle, horiz_var)
    lat_scale = omega_scale_for_bundle(lat_bundle, horiz_var)
    return lon_bundle, lat_bundle, lon_scale, lat_scale


elnino_lon_sections, elnino_lat_sections, elnino_lon_scale, elnino_lat_scale = build_section_bundle(composites['elnino'], 'u')
lanina_lon_sections, lanina_lat_sections, lanina_lon_scale, lanina_lat_scale = build_section_bundle(composites['lanina'], 'v')

lon_levels = symmetric_levels(
    elnino_lon_sections['full']['w'].values,
    elnino_lon_sections['past']['w'].values,
    elnino_lon_sections['recent']['w'].values,
    elnino_lon_sections['diff']['w'].values,
    lanina_lon_sections['full']['w'].values,
    lanina_lon_sections['past']['w'].values,
    lanina_lon_sections['recent']['w'].values,
    lanina_lon_sections['diff']['w'].values,
    n=21,
)
lat_levels = symmetric_levels(
    elnino_lat_sections['full']['w'].values,
    elnino_lat_sections['past']['w'].values,
    elnino_lat_sections['recent']['w'].values,
    elnino_lat_sections['diff']['w'].values,
    lanina_lat_sections['full']['w'].values,
    lanina_lat_sections['past']['w'].values,
    lanina_lat_sections['recent']['w'].values,
    lanina_lat_sections['diff']['w'].values,
    n=21,
)

section_shapes = {
    'elnino_lon': {k: dict(v.sizes) for k, v in elnino_lon_sections.items()},
    'elnino_lat': {k: dict(v.sizes) for k, v in elnino_lat_sections.items()},
    'lanina_lon': {k: dict(v.sizes) for k, v in lanina_lon_sections.items()},
    'lanina_lat': {k: dict(v.sizes) for k, v in lanina_lat_sections.items()},
}

djf_ds


# 4. Plot Composite


### A. Analisis El Nino


#### plot komposit longitude-pressure


In [ ]:
elnino_titles = {
    'full': 'Komposit Sirkulasi Vertikal El Nino DJF 1981-2025\nLongitude-pressure 10S-10N | Quiver: u dan -w dikali skala otomatis',
    'past': 'Komposit Sirkulasi Vertikal El Nino DJF 1981-2006\nLongitude-pressure 10S-10N | Quiver: u dan -w dikali skala otomatis',
    'recent': 'Komposit Sirkulasi Vertikal El Nino DJF 2007-2025\nLongitude-pressure 10S-10N | Quiver: u dan -w dikali skala otomatis',
    'diff': 'Selisih Komposit Sirkulasi Vertikal El Nino DJF P2 - P1\nLongitude-pressure 10S-10N | Quiver: u dan -w dikali skala otomatis',
}

for period_key in ['full', 'past', 'recent', 'diff']:
    fig, ax = plt.subplots(figsize=(12, 5))
    cf = plot_vertical_section(
        ax,
        elnino_lon_sections[period_key],
        horiz_var='u',
        title=elnino_titles[period_key],
        x_dim='lon',
        levels=lon_levels,
        omega_scale=elnino_lon_scale,
        x_ticks=np.arange(60, 271, 30),
        x_label='Longitude (°E; 270°E = 90°W)',
        quiver_scale=55,
    )
    cb = fig.colorbar(cf, ax=ax, pad=0.02, aspect=30)
    cb.set_label('Anomali omega (Pa s$^{-1}$)')
    plt.tight_layout()
    plt.show()



#### plot komposit latitude-pressure


In [ ]:
for period_key in ['full', 'past', 'recent', 'diff']:
    fig, ax = plt.subplots(figsize=(12, 5))
    cf = plot_vertical_section(
        ax,
        elnino_lat_sections[period_key],
        horiz_var='v',
        title=elnino_titles[period_key].replace('Longitude-pressure 10S-10N', 'Latitude-pressure 10S-10N'),
        x_dim='lat',
        levels=lat_levels,
        omega_scale=elnino_lat_scale,
        x_ticks=np.arange(-10, 11, 5),
        x_label='Latitude (°)',
        quiver_scale=55,
    )
    cb = fig.colorbar(cf, ax=ax, pad=0.02, aspect=30)
    cb.set_label('Anomali omega (Pa s$^{-1}$)')
    plt.tight_layout()
    plt.show()



### B. Analisis La Nina


#### plot komposit longitude-pressure


In [ ]:
lanina_titles = {
    'full': 'Komposit Sirkulasi Vertikal La Nina DJF 1981-2025\nLongitude-pressure 10S-10N | Quiver: v dan -w dikali skala otomatis',
    'past': 'Komposit Sirkulasi Vertikal La Nina DJF 1981-2006\nLongitude-pressure 10S-10N | Quiver: v dan -w dikali skala otomatis',
    'recent': 'Komposit Sirkulasi Vertikal La Nina DJF 2007-2025\nLongitude-pressure 10S-10N | Quiver: v dan -w dikali skala otomatis',
    'diff': 'Selisih Komposit Sirkulasi Vertikal La Nina DJF P2 - P1\nLongitude-pressure 10S-10N | Quiver: v dan -w dikali skala otomatis',
}

for period_key in ['full', 'past', 'recent', 'diff']:
    fig, ax = plt.subplots(figsize=(12, 5))
    cf = plot_vertical_section(
        ax,
        lanina_lon_sections[period_key],
        horiz_var='v',
        title=lanina_titles[period_key],
        x_dim='lon',
        levels=lon_levels,
        omega_scale=lanina_lon_scale,
        x_ticks=np.arange(60, 271, 30),
        x_label='Longitude (°E; 270°E = 90°W)',
        quiver_scale=55,
    )
    cb = fig.colorbar(cf, ax=ax, pad=0.02, aspect=30)
    cb.set_label('Anomali omega (Pa s$^{-1}$)')
    plt.tight_layout()
    plt.show()



#### plot komposit latitude-pressure


In [ ]:
for period_key in ['full', 'past', 'recent', 'diff']:
    fig, ax = plt.subplots(figsize=(12, 5))
    cf = plot_vertical_section(
        ax,
        lanina_lat_sections[period_key],
        horiz_var='v',
        title=lanina_titles[period_key].replace('Longitude-pressure 10S-10N', 'Latitude-pressure 10S-10N'),
        x_dim='lat',
        levels=lat_levels,
        omega_scale=lanina_lat_scale,
        x_ticks=np.arange(-10, 11, 5),
        x_label='Latitude (°)',
        quiver_scale=55,
    )
    cb = fig.colorbar(cf, ax=ax, pad=0.02, aspect=30)
    cb.set_label('Anomali omega (Pa s$^{-1}$)')
    plt.tight_layout()
    plt.show()



## Summary

Final inspection outputs are collected below for quick review and copy-paste use.


In [ ]:
print('El Nino years (full):', event_years['elnino']['full'])
print('El Nino years (past):', event_years['elnino']['past'])
print('El Nino years (recent):', event_years['elnino']['recent'])
print('La Nina years (full):', event_years['lanina']['full'])
print('La Nina years (past):', event_years['lanina']['past'])
print('La Nina years (recent):', event_years['lanina']['recent'])

print('\nProcessed cross-section field dimensions:')
for name, shapes in section_shapes.items():
    print(name, shapes)
